# Hindi / Bangla Handwritten Text Segmentation

Line segmentation via **horizontal projection profiling** and word segmentation via **vertical projection profiling**.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from segmentation.io import load_image, save_image, prepare_output_dirs
from segmentation.preprocess import preprocess, to_grayscale, denoise
from segmentation.profiles import horizontal_profile, vertical_profile, smooth_profile
from segmentation.lines import detect_lines, crop_lines
from segmentation.headline import attenuate_headline, detect_headline_band
from segmentation.words import detect_words, crop_words
from segmentation.visualize import annotate_image

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Load Input Image

Replace the path below with your own handwritten Hindi/Bangla JPG.

In [ ]:
INPUT_PATH = "../data/samples/synthetic_test.jpg"   # <-- change this
OUTPUT_DIR = "../outputs/notebook_run"

img = load_image(INPUT_PATH)
plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis("off")
plt.show()

## 2. Preprocessing

Convert to grayscale, denoise, binarize (Otsu), remove small noise, deskew, and crop margins.

In [ ]:
binary, gray, y_off, x_off = preprocess(img, binarize_method="otsu", do_deskew=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Grayscale + denoised")
axes[2].imshow(binary, cmap="gray")
axes[2].set_title("Binary ink mask (cropped)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Binary shape: {binary.shape}")
print(f"Crop offsets: y={y_off}, x={x_off}")

## 3. Horizontal Projection Profile (Line Segmentation)

$H(y) = \sum_x I_{\text{ink}}(x, y)$

Valleys in this profile correspond to gaps between text lines.

In [ ]:
hp = horizontal_profile(binary)
hp_smooth = smooth_profile(hp, window=5)

line_bounds, cc_labels, line_ccs = detect_lines(binary, smooth_window=5)
print(f"Detected {len(line_bounds)} lines: {line_bounds}")

fig, axes = plt.subplots(1, 2, figsize=(14, max(5, binary.shape[0] / 100)))

axes[0].imshow(binary, cmap="gray")
axes[0].set_title("Binary with line boundaries")
for y0, y1 in line_bounds:
    axes[0].axhline(y0, color="lime", linewidth=1)
    axes[0].axhline(y1, color="red", linewidth=1)

axes[1].plot(hp_smooth, np.arange(len(hp_smooth)), color="steelblue")
axes[1].invert_yaxis()
axes[1].set_xlabel("Ink pixel count")
axes[1].set_ylabel("Row (y)")
axes[1].set_title("Horizontal projection profile")
for y0, y1 in line_bounds:
    axes[1].axhline(y0, color="lime", linewidth=0.8, linestyle="--")
    axes[1].axhline(y1, color="red", linewidth=0.8, linestyle="--")

plt.tight_layout()
plt.show()

## 4. Cropped Lines

In [ ]:
line_crops_bin = crop_lines(binary, line_bounds, cc_labels, line_ccs)

fig, axes = plt.subplots(len(line_crops_bin), 1, figsize=(14, 2 * len(line_crops_bin)))
if len(line_crops_bin) == 1:
    axes = [axes]
for i, lc in enumerate(line_crops_bin):
    axes[i].imshow(lc, cmap="gray")
    axes[i].set_title(f"Line {i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 5. Headline Attenuation

Before vertical profiling, the shirorekha / matra headline is detected and attenuated
so that inter-word gaps become visible in the vertical projection.

In [ ]:
sample_line = line_crops_bin[0]
headline_band = detect_headline_band(sample_line)
attenuated = attenuate_headline(sample_line)

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].imshow(sample_line, cmap="gray")
axes[0].set_title("Original line")
if headline_band:
    axes[0].axhline(headline_band[0], color="cyan", linewidth=1)
    axes[0].axhline(headline_band[1], color="cyan", linewidth=1)
axes[1].imshow(attenuated, cmap="gray")
axes[1].set_title("After headline attenuation")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Headline band: {headline_band}")

## 6. Vertical Projection Profile (Word Segmentation)

$V(x) = \sum_y I_{\text{ink}}(x, y)$

Valleys correspond to gaps between words within each line.

In [ ]:
words_per_line = []
for i, lc in enumerate(line_crops_bin):
    wb = detect_words(lc, smooth_window=5)
    words_per_line.append(wb)
    print(f"Line {i}: {len(wb)} words -> {wb}")

total_words = sum(len(w) for w in words_per_line)
print(f"\nTotal words detected: {total_words}")

In [ ]:
for line_idx in range(min(3, len(line_crops_bin))):
    lc = line_crops_bin[line_idx]
    att = attenuate_headline(lc)
    vp = vertical_profile(att)
    vp_smooth = smooth_profile(vp, window=5)
    wb = words_per_line[line_idx]

    fig, axes = plt.subplots(2, 1, figsize=(14, 4))
    axes[0].imshow(lc, cmap="gray")
    axes[0].set_title(f"Line {line_idx}")
    for x0, x1 in wb:
        axes[0].axvline(x0, color="cyan", linewidth=0.8)
        axes[0].axvline(x1, color="orange", linewidth=0.8)

    axes[1].plot(vp_smooth, color="steelblue")
    axes[1].set_title(f"Vertical projection profile – Line {line_idx}")
    axes[1].set_xlabel("Column (x)")
    axes[1].set_ylabel("Ink count")
    for x0, x1 in wb:
        axes[1].axvline(x0, color="cyan", linewidth=0.8, linestyle="--")
        axes[1].axvline(x1, color="orange", linewidth=0.8, linestyle="--")
    plt.tight_layout()
    plt.show()

## 7. Annotated Image

In [ ]:
annotated = annotate_image(img, line_bounds, words_per_line, y_offset=y_off, x_offset=x_off)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("Segmentation result: green = lines, red = words")
plt.axis("off")
plt.show()

## 8. Save Outputs

In [ ]:
dirs = prepare_output_dirs(OUTPUT_DIR)

# Save annotated
save_image(annotated, os.path.join(OUTPUT_DIR, "annotated.png"))

# Save line crops
orig_region = img[y_off:y_off + binary.shape[0], x_off:x_off + binary.shape[1]]
line_crops_orig = crop_lines(binary, line_bounds, original=orig_region)
for i, lc in enumerate(line_crops_orig):
    save_image(lc, dirs["lines"] / f"line_{i:03d}.png")

# Save word crops
for line_idx, (lc_bin, lc_orig, wb) in enumerate(zip(line_crops_bin, line_crops_orig, words_per_line)):
    wc = crop_words(lc_bin, wb, original_line=lc_orig)
    for w_idx, w_img in enumerate(wc):
        save_image(w_img, dirs["words"] / f"line_{line_idx:03d}_word_{w_idx:03d}.png")

print(f"Saved {len(line_bounds)} lines and {total_words} words to {OUTPUT_DIR}")